# Using Psychological Factors to Predict League of Legends Player Rank

Investigating if using a classification model to predict the rank of a league of legends player based on psychological factors.

## 1. Data Loading & Initial Audit

Before cleaning or modeling, I loaded and explored the raw dataset. The questions I had were as follows:
1. What was the structural scale of the data?
1. Are there missing values located and how severe are they?
1. What do the distributions, scales, and extreme values look like across our features?

This data helped me figure out the optimal data classification split (rank 0 or rank 1), and how I needed to approach the pre-processing of this dataset

In [67]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/GamingStudy_data.csv', encoding='latin1')

Cell below investigates the raw dataset. It contains of **13,464 rows and 55 columns** of mixed data types (`int64`, `float64`, and `object/str`)

In [68]:
print(df.info())

<class 'pandas.DataFrame'>
RangeIndex: 13464 entries, 0 to 13463
Data columns (total 55 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       13464 non-null  int64  
 1   Zeitstempel      13464 non-null  float64
 2   GAD1             13464 non-null  int64  
 3   GAD2             13464 non-null  int64  
 4   GAD3             13464 non-null  int64  
 5   GAD4             13464 non-null  int64  
 6   GAD5             13464 non-null  int64  
 7   GAD6             13464 non-null  int64  
 8   GAD7             13464 non-null  int64  
 9   GADE             12815 non-null  str    
 10  SWL1             13464 non-null  int64  
 11  SWL2             13464 non-null  int64  
 12  SWL3             13464 non-null  int64  
 13  SWL4             13464 non-null  int64  
 14  SWL5             13464 non-null  int64  
 15  Game             13464 non-null  str    
 16  Platform         13464 non-null  str    
 17  Hours            13434 

Check missing values across all features to check their severity. 21,709 missing entries are identified. Columns like `highestleague` are completely empty (0 non-null values), while `League`, `Degree`, and individual `SPIN` anxiety metrics have varying data gaps.

In [69]:
print(df.isnull().sum()[df.isnull().sum() > 0])
print(df.isnull().sum().sum())

GADE                 649
Hours                 30
League              1852
highestleague      13464
streams              100
SPIN1                124
SPIN2                154
SPIN3                140
SPIN4                159
SPIN5                166
SPIN6                156
SPIN7                138
SPIN8                144
SPIN9                158
SPIN10               160
SPIN11               187
SPIN12               168
SPIN13               187
SPIN14               156
SPIN15               147
SPIN16               147
SPIN17               175
Narcissism            23
Work                  38
Degree              1577
Reference             15
accept               414
SPIN_T               650
Residence_ISO3       110
Birthplace_ISO3      121
dtype: int64
21709


Investigates the data for each column. 3 columns are reduldant(Unnamed, Zeitstempel and highestleague). Hours contains an impossible max value

There are redundant columns such as `Unnamed: 0`, `Zeitstempel`, and `highestleague`. The `Hours` feature has an incredibly extreme outlier with a maximum value of 8,000 hours per week (which is physically impossible given 168 hours in a week)

In [70]:
print(df.describe().T[['count', 'mean', 'std', 'min', 'max']])

                 count          mean          std           min           max
Unnamed: 0     13464.0   7096.839201  4114.478220      1.000000  14250.000000
Zeitstempel    13464.0  42054.841222     0.272948  42052.004366  42058.363746
GAD1           13464.0      0.860963     0.926542      0.000000      3.000000
GAD2           13464.0      0.673351     0.915724      0.000000      3.000000
GAD3           13464.0      0.965761     0.982776      0.000000      3.000000
GAD4           13464.0      0.724079     0.921971      0.000000      3.000000
GAD5           13464.0      0.488042     0.837014      0.000000      3.000000
GAD6           13464.0      0.911022     0.931168      0.000000      3.000000
GAD7           13464.0      0.588755     0.894408      0.000000      3.000000
SWL1           13464.0      3.720440     1.736264      1.000000      7.000000
SWL2           13464.0      4.602050     1.696276      1.000000      7.000000
SWL3           13464.0      4.345440     1.809430      1.000000 

Analyses the free text input for the league. Will need to standardise these into a clean binary target variable

In [71]:
print(df[df['Game'] == 'League of Legends']['League'].value_counts().head(50))

League
Gold          845
Silver        592
Platinum      545
Diamond       421
gold          277
Unranked      235
Diamond 5     208
silver        195
Gold V        190
Gold 1        189
Gold 3        187
Gold 5        183
Silver 1      182
Silver 2      166
Gold 2        160
platinum      142
Silver 3      137
Bronze        127
diamond       117
Platinum 3    117
Gold 4        116
Silver 4      110
Platinum 2    109
Platinum 1    107
unranked      103
Platinum 5    100
Diamond V      97
Silver 5       94
Platinum 4     80
Platinum V     80
Diamond 3      77
Diamond 4      74
Gold IV        70
gold 5         64
Silver III     64
Diamond 1      61
Gold III       59
Bronze 1       55
Plat 5         53
Silver IV      50
gold 3         50
Silver V       48
Challenger     47
Silver I       47
Plat           46
Diamond 2      45
gold 2         45
Bronze 2       45
diamond 5      45
gold 1         44
Name: count, dtype: int64


## 2. Data Pre-Processing

Before putting anything into the training, the data must first be processed to only contain clean and relevant data.

This pre-processing code can be divided into **three** parts.

1. **Classification Target Mapping:** Convert messy free-text ranks into `League_Binary`
1. **Data Filtering:** Remove physically impossible datapoints and unnecessary metadata for predicting the target variable
1. **Data Verification:** Run another data audit after processing the dataframe to ensure nothing was missed

### Part 1: Map Free-Text to Binary Rank

This is the most important (and complex) step of the pre-processing procedure. The mapping function accounts for variations in spelling and casing, though edge cases exist (e.g, free-text entries citing past ranks like "Diamond 3 last season, placements currently"). The current logic matches the first recognizable tier string found. Will make this function more robust in future.

In [72]:
import re

def map_league_to_binary(league_str):
    if pd.isna(league_str):
        return np.nan

    text = re.sub(r"[^a-z0-9]+", " ", str(league_str).lower()).strip()

    higher_tiers = r"\b(?:platinum|plat|platinium|platin|platnium|platium|platnum|platinuim|platunum|platinumm|platine|platina|pplatinum|plarinum|plantinum|diamond|dia|diamomd|diaomnd|diamont|master|masters|challenger|challenjour|challeneger|charrenjour|challenged)(?:\s*(?:[1-5]|i{1,3}|iv|v))?\b"
    lower_tiers = r"\b(?:bronze|bronce|broze|silver|sliver|siver|silber|sil|gold|glod|golld|goled|golf)(?:\s*(?:[1-5]|i{1,3}|iv|v))?\b"
    division = r"\s*(?:[1-5]|i{1,3}|iv|v)\b"
    high_abbreviations = rf"\b(?:d|p|m|gm){division}"
    low_abbreviations = rf"\b(?:b|s|g){division}"

    if re.search(higher_tiers, text) or re.search(high_abbreviations, text):
        return 1

    if re.search(lower_tiers, text) or re.search(low_abbreviations, text):
        return 0

    return np.nan

In [73]:
df_lol = df[df['Game'] == 'League of Legends']

df_lol['League_Binary'] = df_lol['League'].apply(map_league_to_binary)

### Part 2: Data Filtering

Clean the dataset by removing physically impossible values. Remove duplicate rows and remove unecessary dataset to prevent noise in the model. Also removed rows with missing values. May in future use imputation to predict values, but for now I will just get the minimum working version

In [74]:
df_lol = df_lol[df_lol['Hours'] <= 112]

df_lol = df_lol.drop_duplicates().copy()

cols_to_drop = [
    'Unnamed: 0', 'Zeitstempel', 'highestleague', 'GADE', 'Game', 'Platform', 
    'streams', 'earnings', 'whyplay', 'League', 'Gender', 'Work', 'Degree', 
    'Birthplace', 'Residence', 'Reference', 'Playstyle', 'accept', 
    'Residence_ISO3', 'Birthplace_ISO3'
]

cols_to_drop_existing = [col for col in cols_to_drop if col in df_lol.columns]
df_clean = df_lol.drop(columns=cols_to_drop_existing).copy()

df_clean = df_clean.dropna().copy()

### Part 3: Post Processing Data Audit

For readability, I put this part at the very end. During development I ran the audit iteratively to catch mistakes made with processing early.

In [75]:
print(df_clean.info())

print(df_clean.describe().T[['count', 'mean', 'std', 'min', 'max']])

<class 'pandas.DataFrame'>
Index: 9066 entries, 11 to 13458
Data columns (total 36 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   GAD1           9066 non-null   int64  
 1   GAD2           9066 non-null   int64  
 2   GAD3           9066 non-null   int64  
 3   GAD4           9066 non-null   int64  
 4   GAD5           9066 non-null   int64  
 5   GAD6           9066 non-null   int64  
 6   GAD7           9066 non-null   int64  
 7   SWL1           9066 non-null   int64  
 8   SWL2           9066 non-null   int64  
 9   SWL3           9066 non-null   int64  
 10  SWL4           9066 non-null   int64  
 11  SWL5           9066 non-null   int64  
 12  Hours          9066 non-null   float64
 13  SPIN1          9066 non-null   float64
 14  SPIN2          9066 non-null   float64
 15  SPIN3          9066 non-null   float64
 16  SPIN4          9066 non-null   float64
 17  SPIN5          9066 non-null   float64
 18  SPIN6          9066 no

## 3. Data Split & Baseline

Seperates feature data from the target variable

In [76]:
x = df_clean.drop(columns=['League_Binary'])
y = df_clean['League_Binary']

80/20 train-test split, 80% of data will be used for training while the other 20% will be for testing. Because target variable has a class imbalance, need use stratification. Need to ensure the training and testing sets maintain the same proportion of target classes as the overall dataset

In [77]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    x, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y
)

Create the baseline, create the dummy classifier. Fit the dummy model on the training data and check how it preforms on the test data

In [78]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, classification_report

baseline = DummyClassifier(strategy='most_frequent', random_state=42)
baseline.fit(X_train, y_train)

y_pred_baseline = baseline.predict(X_test)

print(accuracy_score(y_test, y_pred_baseline))
print(classification_report(y_test, y_pred_baseline, zero_division=0))

0.5771775082690187
              precision    recall  f1-score   support

         0.0       0.58      1.00      0.73      1047
         1.0       0.00      0.00      0.00       767

    accuracy                           0.58      1814
   macro avg       0.29      0.50      0.37      1814
weighted avg       0.33      0.58      0.42      1814



# Milestone 3

In [79]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

linear_svc = make_pipeline(
    StandardScaler(),
    LinearSVC(random_state=42, max_iter=10000)
)

In [80]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

svc_scores = cross_validate(
    linear_svc,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"]
)

for metric in ["accuracy", "precision", "recall", "f1"]:
    scores = svc_scores[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.3f} (+/- {scores.std():.3f})")

accuracy: 0.617 (+/- 0.008)
precision: 0.589 (+/- 0.018)
recall: 0.316 (+/- 0.013)
f1: 0.411 (+/- 0.014)


In [85]:
from sklearn.model_selection import GridSearchCV

svc_search = GridSearchCV(
    estimator=linear_svc,
    param_grid={
        "linearsvc__C": [0.01, 0.1, 1, 10, 100],
        "linearsvc__class_weight": [None, "balanced"],
    },
    scoring="f1",
    cv=cv,
    n_jobs=-1,
)

svc_search.fit(X_train, y_train)

print("Best CV F1:", svc_search.best_score_)
print("Best parameters:", svc_search.best_params_)

Best CV F1: 0.5425351906396759
Best parameters: {'linearsvc__C': 0.1, 'linearsvc__class_weight': 'balanced'}


In [87]:
linear_svc.fit(X_train, y_train)
svc_predictions = linear_svc.predict(X_test)

print(classification_report(y_test, svc_predictions, zero_division=0))

              precision    recall  f1-score   support

         0.0       0.64      0.85      0.73      1047
         1.0       0.62      0.34      0.44       767

    accuracy                           0.63      1814
   macro avg       0.63      0.59      0.58      1814
weighted avg       0.63      0.63      0.60      1814



In [88]:
best_svc = svc_search.best_estimator_
svc_predictions = best_svc.predict(X_test)

print(classification_report(y_test, svc_predictions, zero_division=0))

              precision    recall  f1-score   support

         0.0       0.65      0.63      0.64      1047
         1.0       0.52      0.55      0.53       767

    accuracy                           0.59      1814
   macro avg       0.59      0.59      0.59      1814
weighted avg       0.60      0.59      0.60      1814



In [82]:
from sklearn.tree import DecisionTreeClassifier

decision_tree = DecisionTreeClassifier(max_depth=5, random_state=42)

tree_scores = cross_validate(
    decision_tree,
    X_train,
    y_train,
    cv=cv,
    scoring=["accuracy", "precision", "recall", "f1"]
)

for metric in ["accuracy", "precision", "recall", "f1"]:
    scores = tree_scores[f"test_{metric}"]
    print(f"{metric}: {scores.mean():.3f} (+/- {scores.std():.3f})")

accuracy: 0.597 (+/- 0.010)
precision: 0.545 (+/- 0.021)
recall: 0.298 (+/- 0.023)
f1: 0.385 (+/- 0.023)


In [83]:
decision_tree.fit(X_train, y_train)
tree_predictions = decision_tree.predict(X_test)

print(classification_report(y_test, tree_predictions, zero_division=0))

              precision    recall  f1-score   support

         0.0       0.63      0.81      0.71      1047
         1.0       0.58      0.36      0.45       767

    accuracy                           0.62      1814
   macro avg       0.61      0.59      0.58      1814
weighted avg       0.61      0.62      0.60      1814

